# 14 · Decision tree

Compare pre-pruning and cost-complexity pruning for one class-weighted decision tree. Only the frozen train and validation tabular archives are read.


In [1]:
from pathlib import Path
import sys,json,time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier,plot_tree
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd();sys.path.insert(0,str(ROOT/'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics,select_f1_threshold,miss_rate_at_false_alarm_rate
cfg=load_project_config(ROOT/'config.yaml');P=cfg['P']


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Frozen inputs

Trees do not require scaling. Train class weights come from the estimator's `balanced` option.


In [2]:
tr=load_tabular(P,'train');va=load_tabular(P,'val')
Xtr=np.asarray(tr['X'],dtype=np.float64);ytr=np.asarray(tr['y'],dtype=np.int8);Xva=np.asarray(va['X'],dtype=np.float64);yva=np.asarray(va['y'],dtype=np.int8);names=np.asarray(tr['feature_names']).astype(str)
assert Xtr.shape==(7607,84) and Xva.shape==(1510,84)
assert np.array_equal(tr['clip_id'],pd.read_csv(P['splits']/'train.csv',keep_default_na=False).clip_id)
assert np.array_equal(va['clip_id'],pd.read_csv(P['splits']/'val.csv',keep_default_na=False).clip_id)


## Pre-pruned grid and pruning path

Score 18 Gini configurations, one entropy variant at the best Gini pair, and up to 20 log-spaced cost-complexity alphas computed from train only. Non-default choices must improve validation F1 by at least 0.01; ties favor fewer nodes.


In [3]:
t0=time.perf_counter();rows=[];models={}
configs=[{'criterion':'gini','max_depth':d,'min_samples_leaf':leaf} for d in (2,3,4,6,8,None) for leaf in (1,5,20)]
def score_tree(model,kind,hp):
 s=model.predict_proba(Xva)[:,1];th=select_f1_threshold(yva,s);m=classification_metrics(yva,s,threshold=th)
 row={'kind':kind,**hp,'threshold':float(th),'val_f1':m['f1'],'val_roc_auc':m['roc_auc'],'far':m['false_alarm_rate'],'miss':m['miss_rate'],'nodes':int(model.tree_.node_count),'depth':int(model.tree_.max_depth)}
 rows.append(row);models[len(rows)-1]=model
for hp in configs:
 model=DecisionTreeClassifier(**hp,class_weight='balanced',random_state=int(cfg['seed'])).fit(Xtr,ytr);score_tree(model,'prepruned',hp)
best_gini=max((i for i,r in enumerate(rows) if r['kind']=='prepruned'),key=lambda i:rows[i]['val_f1'])
bg=rows[best_gini]; hp={k:bg[k] for k in ('max_depth','min_samples_leaf')}
model=DecisionTreeClassifier(criterion='entropy',**hp,class_weight='balanced',random_state=int(cfg['seed'])).fit(Xtr,ytr);score_tree(model,'entropy_variant',{'criterion':'entropy',**hp})
full=DecisionTreeClassifier(criterion='gini',class_weight='balanced',random_state=int(cfg['seed'])).fit(Xtr,ytr)
path=full.cost_complexity_pruning_path(Xtr,ytr);alphas=np.unique(path.ccp_alphas)
if len(alphas)>20: alphas=np.unique(np.geomspace(max(alphas[1],1e-12),alphas[-1],19).tolist()+[0.0])
for alpha in alphas:
 model=DecisionTreeClassifier(criterion='gini',ccp_alpha=float(alpha),class_weight='balanced',random_state=int(cfg['seed'])).fit(Xtr,ytr);score_tree(model,'pruned',{'ccp_alpha':float(alpha)})
results=pd.DataFrame(rows); pre=results[results.kind!='pruned'].sort_values(['val_f1','nodes'],ascending=[False,True]).iloc[0];pruned=results[results.kind=='pruned'].sort_values(['val_f1','nodes'],ascending=[False,True]).iloc[0]
base=max(pre.val_f1,pruned.val_f1); candidates=[pre,pruned]; chosen_row=sorted(candidates,key=lambda r:(-(r.val_f1 if r.val_f1>=base-.01 else base),r.nodes))[0]
if chosen_row.val_f1 < base-.01: chosen_row=pre if pre.val_f1>=pruned.val_f1 else pruned
chosen_row=results.loc[(results.kind==chosen_row.kind)&(results.nodes==chosen_row.nodes)&(results.val_f1==chosen_row.val_f1)].iloc[0]
chosen_idx=int(chosen_row.name);model=models[chosen_idx]; score=model.predict_proba(Xva)[:,1];threshold=select_f1_threshold(yva,score);metrics=classification_metrics(yva,score,threshold=threshold)
print('Fits:',len(results),'pruning alphas:',len(alphas),'elapsed:',round(time.perf_counter()-t0,2),'s');print('Frozen:',chosen_row.to_dict())


Fits: 39 pruning alphas: 20 elapsed: 43.38 s
Frozen: {'kind': 'pruned', 'criterion': nan, 'max_depth': nan, 'min_samples_leaf': nan, 'threshold': 0.8950026770704234, 'val_f1': 0.7417582417582418, 'val_roc_auc': 0.9297542070372259, 'far': 0.03320754716981132, 'miss': 0.2702702702702703, 'nodes': 49, 'depth': 8, 'ccp_alpha': 0.0022271107275436026}


## Diagnostics and freeze

Show the validation F1 against tree size, the top tree levels, and impurity feature importance. The held-out test split is not opened.


In [4]:
miss1,thr1=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.01);miss5,thr5=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.05)
FIG=ROOT/'results'/'figures';TABLES=ROOT/'results'/'tables';MODELS=ROOT/'results'/'models';CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS):d.mkdir(parents=True,exist_ok=True)
results.to_csv(TABLES/'14_decision_tree_grid.csv',index=False)
imp=pd.DataFrame({'feature':names,'importance':model.feature_importances_}).sort_values('importance',ascending=False);imp.to_csv(TABLES/'14_decision_tree_importance.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5));ax.scatter(results.nodes,results.val_f1,c=(results.kind=='pruned'));ax.set(xlabel='Tree nodes',ylabel='Validation F1',title='Decision tree size and validation score');fig.tight_layout();fig.savefig(FIG/'14_decision_tree_size.png',dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(18,9));plot_tree(model,feature_names=names,class_names=['Unknown','Drone'],max_depth=3,filled=True,fontsize=7,ax=ax);fig.tight_layout();fig.savefig(FIG/'14_decision_tree_top.png',dpi=160);plt.close(fig)
card={'model':'decision_tree','representation':'84 tabular features','best_hyperparameters':{k:(chosen_row[k].item() if hasattr(chosen_row[k],'item') else chosen_row[k]) for k in ('kind','ccp_alpha') if k in chosen_row.index and pd.notna(chosen_row[k])},'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5)},'seeds':[int(cfg['seed'])],'train_time_s':float(time.perf_counter()-t0),'diagnostics':{'nodes':int(model.tree_.node_count),'depth':int(model.tree_.max_depth)},'decision_log_ids':['D044','D045']}
fh=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash'];joblib.dump({'model':model,'feature_names':names.tolist(),'threshold':float(threshold),'threshold_rule':'validation_f1_max_ties_higher','front_end_hash':fh},MODELS/'14_decision_tree.joblib');(CARDS/'14_decision_tree.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8');print('Validation:',metrics)


Validation: {'accuracy': 0.937748344370861, 'precision': 0.7541899441340782, 'recall': 0.7297297297297297, 'f1': 0.7417582417582418, 'roc_auc': 0.9297542070372259, 'false_alarm_rate': 0.03320754716981132, 'miss_rate': 0.2702702702702703, 'confusion_matrix': [[1281, 44], [50, 135]]}
